# 00 — Introducción y visión del proyecto

**Audiencia:** tutor de tesis / personas sin conocimiento previo de packing.

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

| Grupo benchmark | Instancia | Diferenciación esperada |
|-----------------|-----------|-------------------------|
| 3D-BPP | `showcase_3d_bpp_instance.json` | best_fit ~27 emp. vs extreme_points ~22 |
| Híbrido (misma instancia 3D) | `showcase_3d_bpp_instance.json` | compaction/LS mueven 8–11 piezas |
| Container Loading | `showcase_container_loading_instance.json` | weight_aware ~27 vs single_container ~17 |
| Single Container | `showcase_single_container_instance.json` | best_fit ~13 (94% util) vs first_fit ~16 |
| Cartonization | `showcase_cartonization_instance.json` | BOX_M (~49%) vs BOX_L (~21%) |

## ¿Qué problema resolvemos?

Dado un **catálogo de piezas** (cajas con dimensiones y cantidades) y uno o más **contenedores**,
hay que decidir **dónde colocar cada pieza** (posición y rotación) para:
- Maximizar el aprovechamiento del espacio
- Respetar peso máximo y no solapar piezas
- Reportar qué piezas no cupieron

Eso pertenece a la familia *Cutting and Packing*, con **cuatro formulaciones** en este proyecto:
3D-BPP, Container Loading, Cartonization y Single Container Loading.

## La brecha que identifiqué

Existen muchos algoritmos, pero cada implementación usa **formatos distintos**.
No se pueden comparar de forma justa ni integrar en un espacio de datos.

## Mi propuesta

Plataforma de **servicios** donde cada algoritmo se registra, ejecuta, valida, compara y documenta homogéneamente.
El **benchmark** usa perfiles estándar (`constructive`, `hybrid`, `box_selection`, …) para comparar ≥2 motores sobre la misma instancia.


In [1]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


Proyecto: /home/edmundo/packing-services


In [2]:
import subprocess, sys
from _shared.loaders import load_showcase_instance, SHOWCASE_INSTANCES, BENCHMARK_GROUPS
from packing_services import SERVICE_VERSION
from packing_services.algorithms.registry import get_default_registry
from packing_services.domain.enums import AlgorithmStatus

display.show_problem_overview(load_showcase_instance('3D_BPP'))

registry = get_default_registry()
n_impl = len(registry.list_metadata(status=AlgorithmStatus.IMPLEMENTED))
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', str(ROOT / 'tests'), '--collect-only', '-q',
     '--ignore=tests/test_api.py'],
    capture_output=True, text=True, cwd=ROOT,
)
n_tests = next((int(l.split()[0]) for l in test_run.stdout.splitlines() if 'tests collected' in l), '?')
print(f'\n=== packing-services v{SERVICE_VERSION} ===')
print(f'Algoritmos implementados: {n_impl} | Tests: {n_tests} | Servicios operativos: 5')
print(f'Tipos de problema: {len(SHOWCASE_INSTANCES)} | Grupos de benchmark: {len(BENCHMARK_GROUPS)}')
from _shared.loaders import SHOWCASE_DIFFERENTIATION
import pandas as pd
from IPython.display import display as ipy_display
rows = [{'Grupo': g, 'Diferenciación esperada': SHOWCASE_DIFFERENTIATION[g]} for g in BENCHMARK_GROUPS]
ipy_display(pd.DataFrame(rows).style.hide(axis='index'))


Contenedores disponibles:


ID,Largo,Ancho,Alto,Peso máx,Volumen
C1,100,80,80,500,640000
C2,100,80,80,500,640000


Catálogo de piezas (ítems):


Tipo,Largo,Ancho,Alto,Peso,Cantidad
P1,50,40,30,6,6
P2,60,30,40,7,5
P3,45,45,25,5,4
P4,30,25,55,4,5
P5,20,20,20,2,10


Restricciones activas: non_overlap, containment, allow_rotation, max_weight



=== packing-services v0.1.0 ===
Algoritmos implementados: 12 | Tests: ? | Servicios operativos: 5
Tipos de problema: 5 | Grupos de benchmark: 5


Grupo,Diferenciación esperada
3D_BPP,best_fit ~27 emp. (85% util) vs extreme_points ~22 (66%)
3D_HYBRID,misma cantidad empacada; compaction/LS mueven 8–11 piezas
CONTAINER_LOADING,weight_aware ~27 emp. vs single_container ~17 (alta util) vs extreme_points ~22
CARTONIZATION,minimalistas → BOX_M (~49% util) vs largest_feasible → BOX_L (~21%)
SINGLE_CONTAINER_LOADING,best_fit/single ~13 emp. (94% util) vs first_fit ~16 (89%)


In [3]:
print('═══ Vista previa: benchmarks homogéneos (mismas instancias en todos los notebooks) ═══')
for group in BENCHMARK_GROUPS:
    display.run_and_show_showcase_benchmark(group, show_chart=False)


═══ Vista previa: benchmarks homogéneos (mismas instancias en todos los notebooks) ═══
Motores comparables — 3D Bin Packing:


#,Algoritmo,Nombre,Opcional
1,heuristic_3d_bpp_v1,Volume First Candidate Placement,No
2,first_fit_decreasing_3d,First Fit Decreasing 3D,No
3,extreme_points_3d,Extreme Points Heuristic,No
4,best_fit_decreasing_3d,Best Fit Decreasing 3D,No
5,py3dbp_adapter,py3dbp Adapter (baseline externo),Sí


Nota: best_fit ~27 emp. (85% util) vs extreme_points ~22 (66%)
Benchmark — 3D Bin Packing


Posición,Algoritmo,Utilización,Empacados,Sin empacar,Válida,Tiempo (s)
1,best_fit_decreasing_3d,84.8%,27,3,Sí,0.0299
2,first_fit_decreasing_3d,78.3%,25,5,Sí,0.0075
3,heuristic_3d_bpp_v1,78.3%,25,5,Sí,0.0108
4,py3dbp_adapter,78.3%,25,5,Sí,0.0122
5,extreme_points_3d,65.7%,22,8,Sí,0.0163


Ranking 3D-BPP: (1) soluciones válidas; (2) mayor volume_utilization; (3) menor items_unpacked; (4) menor containers_used; (5) menor tiempo.
Motores comparables — 3D Bin Packing — Constructivo + mejora local:


...

Ranking Cartonization: (1) válidas; (2) todos los ítems empacados; (3) mayor volume_utilization; (4) menor tiempo.
Motores comparables — Single Container Loading:


#,Algoritmo,Nombre,Opcional
1,single_container_constructive,Single Container Constructive,No
2,best_fit_decreasing_3d,Best Fit Decreasing 3D,No
3,first_fit_decreasing_3d,First Fit Decreasing 3D,No


Nota: best_fit/single ~13 emp. (94% util) vs first_fit ~16 (89%)
Benchmark — Single Container Loading


Posición,Algoritmo,Utilización,Empacados,Sin empacar,Válida,Tiempo (s)
1,single_container_constructive,94.3%,13,17,Sí,0.0096
2,best_fit_decreasing_3d,94.3%,13,17,Sí,0.0116
3,first_fit_decreasing_3d,89.3%,16,14,Sí,0.0056


Ranking Single Container Loading: (1) válidas; (2) mayor volume_utilization; (3) menor items_unpacked; (4) menor tiempo.


## Puente: informe técnico → código

| Decisión del informe | Evidencia en el repositorio |
|----------------------|----------------------------|
| Priorizar heurísticas constructivas | 4 heurísticas 3D-BPP + py3dbp opcional |
| Mejora local reutilizable | `solution_compaction`, `constructive_plus_local_search` |
| Validador geométrico propio | `PackingValidator` + tests |
| Servicios independientes | 5 servicios con API + perfiles de benchmark |
| Preparación espacio de datos | `GET /api/v1/services` |

**Siguiente:** `01_arquitectura_y_servicios.ipynb`
